In [15]:
# Requires: pip install yfinance pandas

import yfinance as yf
import pandas as pd

# Download XOM data for 2024
xom = yf.download("XOM",
                  start="2024-01-01",
                  end="2025-01-01",
                  auto_adjust=False,
                  progress=False)

# Reset index so Date becomes a column
xom = xom.reset_index()

# Keep the same columns as your AAPL file
xom = xom[["Date", "Open", "High", "Low", "Close", "Adj Close", "Volume"]]

# Save to CSV
xom.to_csv("XOM_2024.csv", index=False)

print("✅ Saved XOM_2024.csv with columns:", xom.columns.tolist())
print(xom.head())


✅ Saved XOM_2024.csv with columns: [('Date', ''), ('Open', 'XOM'), ('High', 'XOM'), ('Low', 'XOM'), ('Close', 'XOM'), ('Adj Close', 'XOM'), ('Volume', 'XOM')]
Price        Date        Open        High         Low       Close  Adj Close  \
Ticker                    XOM         XOM         XOM         XOM        XOM   
0      2024-01-02  100.919998  103.099998  100.849998  102.360001  96.314430   
1      2024-01-03  102.269997  103.620003  101.660004  103.220001  97.123634   
2      2024-01-04  104.080002  104.570000  102.050003  102.320000  96.276794   
3      2024-01-05  103.169998  103.400002  102.129997  102.629997  96.568481   
4      2024-01-08  100.730003  101.040001   98.900002  100.919998  94.959473   

Price     Volume  
Ticker       XOM  
0       23483000  
1       23490800  
2       19395200  
3       15827400  
4       23370100  


In [16]:
import pandas as pd
import numpy as np
import torch
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.model_selection import train_test_split
import re
import string
from sentence_transformers import SentenceTransformer
import nltk



In [17]:
device = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
print(f"Using device: {device}")


Using device: mps


In [18]:
filepath_num = 'XOM_2024.csv'
filepath_sent = 'annotated_sp500_news_2024_XOM.csv'

data_num = pd.read_csv(filepath_num)
data_sent = pd.read_csv(filepath_sent)

In [19]:
data_sent.head()

,date,title,content,sentiment_score
0,02/01/2024,3 Dimensional Mutual Funds for Consistent Returns,"Below, we share with you three top-ranked Dime...",0.11
1,02/01/2024,"ExxonMobil Reportedly Bids Iraq Farewell, Petr...",Exxon Mobil Corp XOM has reportedly exited the...,0.00
2,02/01/2024,Analysts Predict Surprising New 'Magnificent 7...,"Forget the ""Magnificent Seven"" stocks that dro...",0.28
3,02/01/2024,"Down 9% Since The Beginning Of 2023, What Shou...",After a 9% decline since the beginning of 2023...,0.02
4,02/01/2024,This Giant Oil Stock Is My Top Dividend Pick f...,"Exxon's strong dividends, its record of strong...",0.16


In [20]:
data_num.head()

,Date,Open,High,Low,Close,Adj Close,Volume
0,NaN,XOM,XOM,XOM,XOM,XOM,XOM
1,2024-01-02,100.91999816894531,103.0999984741211,100.8499984741211,102.36000061035156,96.3144302368164,23483000
2,2024-01-03,102.2699966430664,103.62000274658203,101.66000366210938,103.22000122070312,97.1236343383789,23490800
3,2024-01-04,104.08000183105469,104.56999969482422,102.05000305175781,102.31999969482422,96.27679443359375,19395200
4,2024-01-05,103.16999816894531,103.4000015258789,102.12999725341797,102.62999725341797,96.5684814453125,15827400


In [21]:
# Check if 'Unnamed: 0' exists in the columns and drop it
if 'Unnamed: 0' in data_sent.columns:
    data_sent = data_sent.drop(columns=['Unnamed: 0'])
data_sent.rename(columns={'Time': 'Date'}, inplace=True)
data_sent.head()

,date,title,content,sentiment_score
0,02/01/2024,3 Dimensional Mutual Funds for Consistent Returns,"Below, we share with you three top-ranked Dime...",0.11
1,02/01/2024,"ExxonMobil Reportedly Bids Iraq Farewell, Petr...",Exxon Mobil Corp XOM has reportedly exited the...,0.00
2,02/01/2024,Analysts Predict Surprising New 'Magnificent 7...,"Forget the ""Magnificent Seven"" stocks that dro...",0.28
3,02/01/2024,"Down 9% Since The Beginning Of 2023, What Shou...",After a 9% decline since the beginning of 2023...,0.02
4,02/01/2024,This Giant Oil Stock Is My Top Dividend Pick f...,"Exxon's strong dividends, its record of strong...",0.16


In [22]:
if 'date' in data_num.columns:
    data_num.rename(columns={'date':'Date'}, inplace=True)
if 'date' in data_sent.columns:
    data_sent.rename(columns={'date':'Date'}, inplace=True)


In [23]:
data_num['Date'] = pd.to_datetime(data_num['Date'])  
# sentiment dates are in dd/mm/YYYY format, so force dayfirst
data_sent['Date']  = pd.to_datetime(data_sent['Date'], format='%d/%m/%Y', dayfirst=True)


In [24]:
# 3) Aggregate sentiment per date
agg_sent = (
    data_sent
    .groupby('Date', as_index=False)['sentiment_score']
    .sum()
    .rename(columns={'sentiment_score':'aggregate_sentiment_score'})
)

# 4) Merge on Date (inner join will only keep dates present in both)
merged = pd.merge(
    data_num,
    agg_sent,
    on='Date',
    how='inner'
)

In [25]:
# 5) Select the columns you need
final_data = merged[['Date','Open','Close','High','Volume','aggregate_sentiment_score']]

# 6) Quick sanity‐check
print(final_data.shape)
print(final_data.head())

(250, 6)
        Date                Open               Close                High  \
0 2024-01-02  100.91999816894531  102.36000061035156   103.0999984741211   
1 2024-01-03   102.2699966430664  103.22000122070312  103.62000274658203   
2 2024-01-04  104.08000183105469  102.31999969482422  104.56999969482422   
3 2024-01-05  103.16999816894531  102.62999725341797   103.4000015258789   
4 2024-01-08   100.7300033569336  100.91999816894531  101.04000091552734   

     Volume  aggregate_sentiment_score  
0  23483000                       0.62  
1  23490800                      -0.26  
2  19395200                       0.78  
3  15827400                       0.64  
4  23370100                       0.04  


In [26]:
import pandas as pd
import numpy as np

# --- 0) Ensure types are correct ---
# If your Date is a string like 'Dec 30 2024', parse it and sort
final_data["Date"] = pd.to_datetime(final_data["Date"], errors="coerce", infer_datetime_format=True)
final_data = final_data.sort_values("Date").reset_index(drop=True)

# Coerce OHLCV to numeric (the TypeError came from string dtypes here)
num_cols = ["Open","High","Close","Adj Close","Volume"]
for c in num_cols:
    if c in final_data.columns:
        final_data[c] = pd.to_numeric(final_data[c], errors="coerce")

# If you have sentiment columns, coerce them too
if "aggregate_sentiment_score" in final_data.columns:
    final_data["aggregate_sentiment_score"] = pd.to_numeric(final_data["aggregate_sentiment_score"], errors="coerce")

# Optional: drop rows with missing core prices
final_data = final_data.dropna(subset=["Open","High","Close"])

# --- 1) Label: next-day return ---
# Choose ONE of the two definitions below.

# (A) Close-to-next-Close return (common):
#final_data["Movement_raw"] = (final_data["Close"].shift(-1) - final_data["Close"]) / final_data["Close"]

# (B) Overnight gap: Close-to-next-Open (uncomment to use this instead)
final_data["Movement_raw"] = (final_data["Open"].shift(-1) - final_data["Close"]) / final_data["Close"]

# IMPORTANT: Do NOT shift() again afterwards. Using shift(-1) already aligns the
# label with "today t" based on information from "t+1".
# Drop last row (no next-day)
final_data = final_data.dropna(subset=["Movement_raw"])

# Binary label: 1 if up, else 0
final_data["Movement"] = (final_data["Movement_raw"] > 0).astype(int)

# --- 2) Daily returns & volatility (features) ---
# Daily return (in percent, like your code)
final_data["Daily_Return"] = final_data["Close"].pct_change() * 100

# Rolling volatility over window_size; then lag it to avoid peeking into the future
window_size = 5
final_data["Volatility"] = final_data["Daily_Return"].rolling(window=window_size).std()
final_data["Volatility_lag1"] = final_data["Volatility"].shift(1)

# --- 3) Sentiment volatility & lags (features) ---
if "aggregate_sentiment_score" in final_data.columns:
    sentiment_window_size = 5
    final_data["sentiment_volatility"] = (
        final_data["aggregate_sentiment_score"]
        .rolling(window=sentiment_window_size)
        .std()
    )
    # Lag sentiment-derived features to avoid leakage
    final_data["sentiment_volatility_lag1"] = final_data["sentiment_volatility"].shift(1)
    final_data["aggregate_sentiment_score_lag1"] = final_data["aggregate_sentiment_score"].shift(1)

# --- 4) Other lagged market features (avoid leakage) ---
final_data["Close_lag1"] = final_data["Close"].shift(1)
final_data["High_lag1"] = final_data["High"].shift(1)
final_data["Volume_lag1"] = final_data["Volume"].shift(1)
final_data["Daily_Return_lag1"] = final_data["Daily_Return"].shift(1)

# --- 5) Final NaN drop for training-ready data ---
needed = ["Movement","Volatility_lag1","Close_lag1","High_lag1","Volume_lag1","Daily_Return_lag1"]
if "sentiment_volatility_lag1" in final_data.columns:
    needed += ["sentiment_volatility_lag1","aggregate_sentiment_score_lag1"]

final_data = final_data.dropna(subset=needed).reset_index(drop=True)

# Peek
print(final_data.head(10))


        Date        Open      Close        High    Volume  \
0 2024-01-10   99.800003  98.690002   99.800003  18206100   
1 2024-01-11   99.040001  98.669998   99.500000  15833400   
2 2024-01-12  100.139999  99.949997  100.650002  18041700   
3 2024-01-16   99.820000  97.690002  100.010002  20235700   
4 2024-01-17   96.599998  96.980003   97.959999  18384000   
5 2024-01-18   97.000000  96.800003   97.089996  20940300   
6 2024-01-19   96.720001  96.949997   97.019997  20088400   
7 2024-01-22   96.699997  96.820000   97.099998  19955900   
8 2024-01-23   96.809998  97.910004   98.500000  15863400   
9 2024-01-24   98.320000  99.599998   99.650002  17330600   

   aggregate_sentiment_score  Movement_raw  Movement  Daily_Return  \
0                       0.49      0.003546         1     -0.983240   
1                      -0.15      0.014898         1     -0.020270   
2                       0.46     -0.001301         0      1.297252   
3                       0.26     -0.011158      

/var/folders/pd/35sqmnx90k31hxsfkmzt115m000cd0/T/ipykernel_48767/3334214582.py:6: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  final_data["Date"] = pd.to_datetime(final_data["Date"], errors="coerce", infer_datetime_format=True)
/var/folders/pd/35sqmnx90k31hxsfkmzt115m000cd0/T/ipykernel_48767/3334214582.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_data["Date"] = pd.to_datetime(final_data["Date"], errors="coerce", infer_datetime_format=True)


In [27]:
final_data.to_csv('merged_XOM.csv')

In [28]:
device = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
print(f"Using device: {device}")


Using device: mps


In [29]:
filepath_num = 'XOM_2024.csv'
filepath_sent = 'annotated_sp500_news_2024_XOM.csv'

data_num = pd.read_csv(filepath_num)
data_sent = pd.read_csv(filepath_sent)

In [30]:
data_sent.head()

,date,title,content,sentiment_score
0,02/01/2024,3 Dimensional Mutual Funds for Consistent Returns,"Below, we share with you three top-ranked Dime...",0.11
1,02/01/2024,"ExxonMobil Reportedly Bids Iraq Farewell, Petr...",Exxon Mobil Corp XOM has reportedly exited the...,0.00
2,02/01/2024,Analysts Predict Surprising New 'Magnificent 7...,"Forget the ""Magnificent Seven"" stocks that dro...",0.28
3,02/01/2024,"Down 9% Since The Beginning Of 2023, What Shou...",After a 9% decline since the beginning of 2023...,0.02
4,02/01/2024,This Giant Oil Stock Is My Top Dividend Pick f...,"Exxon's strong dividends, its record of strong...",0.16


In [31]:
import re
import nltk

# Download the sentence tokenizer only once
nltk.download("punkt_tab")  

from nltk.tokenize import sent_tokenize

# ----------------------------------------
# 1. Minimal cleaning function (unchanged)
# ----------------------------------------
def minimal_clean_text(text: str, to_lowercase: bool = False) -> str:
    """
    Minimal cleaning for transformer-based models:
    1. Remove HTML tags.
    2. Remove/replacing weird characters if needed.
    3. (Optional) Convert to lowercase if you're using an uncased model.
    """
    # 1. Remove HTML tags:
    text = re.sub(r'<.*?>', '', text)

    # 2. Remove or replace unwanted characters (non-ASCII -> space here):
    text = re.sub(r'[^\x00-\x7F]+', ' ', text)

    # 3. (Optional) Lowercasing:
    if to_lowercase:
        text = text.lower()

    # Remove excessive whitespace
    text = " ".join(text.split())

    return text

# --------------------------------------------
# 2. Helper function: remove duplicate sentences
# --------------------------------------------
def remove_duplicate_sentences(text: str) -> str:
    """
    Splits the text into sentences (using NLTK), 
    then removes exact duplicate sentences. 
    Joins back with a space.
    """
    sentences = sent_tokenize(text)
    seen = set()
    unique_sentences = []
    
    for s in sentences:
        s = s.strip()
        if s not in seen:
            unique_sentences.append(s)
            seen.add(s)
            
    return " ".join(unique_sentences)

# ---------------------------------------------
# 3. Clean the content only (exclude the Title)
# ---------------------------------------------
data_sent['cleaned_content'] = data_sent['content'].apply(
    lambda x: minimal_clean_text(x, to_lowercase=False)
)

# ---------------------------------------------
# 4. Remove duplicate sentences from each entry
# ---------------------------------------------
data_sent['cleaned_content'] = data_sent['cleaned_content'].apply(remove_duplicate_sentences)

# (The group-by and merging part has been removed)


[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/skhanna/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [32]:
data_sent = data_sent.rename(columns={'date': 'Date'})

In [33]:
# Step 1: Parse day-first
data_sent['Date'] = pd.to_datetime(data_sent['Date'], dayfirst=True)
# 2. Sort by date in ascending order
data_sent.sort_values(by='Date', inplace=True)
data_sent.head()


,Date,title,content,sentiment_score,cleaned_content
0,2024-01-02,3 Dimensional Mutual Funds for Consistent Returns,"Below, we share with you three top-ranked Dime...",0.11,"Below, we share with you three top-ranked Dime..."
1,2024-01-02,"ExxonMobil Reportedly Bids Iraq Farewell, Petr...",Exxon Mobil Corp XOM has reportedly exited the...,0.00,Exxon Mobil Corp XOM has reportedly exited the...
2,2024-01-02,Analysts Predict Surprising New 'Magnificent 7...,"Forget the ""Magnificent Seven"" stocks that dro...",0.28,"Forget the ""Magnificent Seven"" stocks that dro..."
3,2024-01-02,"Down 9% Since The Beginning Of 2023, What Shou...",After a 9% decline since the beginning of 2023...,0.02,After a 9% decline since the beginning of 2023...
4,2024-01-02,This Giant Oil Stock Is My Top Dividend Pick f...,"Exxon's strong dividends, its record of strong...",0.16,"Exxon's strong dividends, its record of strong..."


In [34]:
if 'Unnamed: 0' in data_sent.columns:
    data_sent = data_sent.drop(columns=['Unnamed: 0'])
data_sent.head()

,Date,title,content,sentiment_score,cleaned_content
0,2024-01-02,3 Dimensional Mutual Funds for Consistent Returns,"Below, we share with you three top-ranked Dime...",0.11,"Below, we share with you three top-ranked Dime..."
1,2024-01-02,"ExxonMobil Reportedly Bids Iraq Farewell, Petr...",Exxon Mobil Corp XOM has reportedly exited the...,0.00,Exxon Mobil Corp XOM has reportedly exited the...
2,2024-01-02,Analysts Predict Surprising New 'Magnificent 7...,"Forget the ""Magnificent Seven"" stocks that dro...",0.28,"Forget the ""Magnificent Seven"" stocks that dro..."
3,2024-01-02,"Down 9% Since The Beginning Of 2023, What Shou...",After a 9% decline since the beginning of 2023...,0.02,After a 9% decline since the beginning of 2023...
4,2024-01-02,This Giant Oil Stock Is My Top Dividend Pick f...,"Exxon's strong dividends, its record of strong...",0.16,"Exxon's strong dividends, its record of strong..."


In [35]:
if 'Unnamed: 0' in data_num.columns:
    data_num = data_num.drop(columns=['Unnamed: 0'])
data_num.head()

,Date,Open,High,Low,Close,Adj Close,Volume
0,NaN,XOM,XOM,XOM,XOM,XOM,XOM
1,2024-01-02,100.91999816894531,103.0999984741211,100.8499984741211,102.36000061035156,96.3144302368164,23483000
2,2024-01-03,102.2699966430664,103.62000274658203,101.66000366210938,103.22000122070312,97.1236343383789,23490800
3,2024-01-04,104.08000183105469,104.56999969482422,102.05000305175781,102.31999969482422,96.27679443359375,19395200
4,2024-01-05,103.16999816894531,103.4000015258789,102.12999725341797,102.62999725341797,96.5684814453125,15827400


In [36]:
# Merge on the 'date' column, keeping only dates present in both dataframes
data_num['Date'] = pd.to_datetime(data_num['Date'])
merged_df = pd.merge(data_num, data_sent, on='Date', how='inner')

merged_df.head()


,Date,Open,High,Low,Close,Adj Close,Volume,title,content,sentiment_score,cleaned_content
0,2024-01-02,100.91999816894531,103.0999984741211,100.8499984741211,102.36000061035156,96.3144302368164,23483000,3 Dimensional Mutual Funds for Consistent Returns,"Below, we share with you three top-ranked Dime...",0.11,"Below, we share with you three top-ranked Dime..."
1,2024-01-02,100.91999816894531,103.0999984741211,100.8499984741211,102.36000061035156,96.3144302368164,23483000,"ExxonMobil Reportedly Bids Iraq Farewell, Petr...",Exxon Mobil Corp XOM has reportedly exited the...,0.00,Exxon Mobil Corp XOM has reportedly exited the...
2,2024-01-02,100.91999816894531,103.0999984741211,100.8499984741211,102.36000061035156,96.3144302368164,23483000,Analysts Predict Surprising New 'Magnificent 7...,"Forget the ""Magnificent Seven"" stocks that dro...",0.28,"Forget the ""Magnificent Seven"" stocks that dro..."
3,2024-01-02,100.91999816894531,103.0999984741211,100.8499984741211,102.36000061035156,96.3144302368164,23483000,"Down 9% Since The Beginning Of 2023, What Shou...",After a 9% decline since the beginning of 2023...,0.02,After a 9% decline since the beginning of 2023...
4,2024-01-02,100.91999816894531,103.0999984741211,100.8499984741211,102.36000061035156,96.3144302368164,23483000,This Giant Oil Stock Is My Top Dividend Pick f...,"Exxon's strong dividends, its record of strong...",0.16,"Exxon's strong dividends, its record of strong..."


In [37]:
merged_df.value_counts

<bound method DataFrame.value_counts of            Date                Open                High                 Low  \
0    2024-01-02  100.91999816894531   103.0999984741211   100.8499984741211   
1    2024-01-02  100.91999816894531   103.0999984741211   100.8499984741211   
2    2024-01-02  100.91999816894531   103.0999984741211   100.8499984741211   
3    2024-01-02  100.91999816894531   103.0999984741211   100.8499984741211   
4    2024-01-02  100.91999816894531   103.0999984741211   100.8499984741211   
...         ...                 ...                 ...                 ...   
1704 2024-12-30  106.30000305175781  106.55999755859375  105.51000213623047   
1705 2024-12-30  106.30000305175781  106.55999755859375  105.51000213623047   
1706 2024-12-30  106.30000305175781  106.55999755859375  105.51000213623047   
1707 2024-12-31  106.16999816894531   107.9000015258789  105.77999877929688   
1708 2024-12-31  106.16999816894531   107.9000015258789  105.77999877929688   

           

In [38]:
merged_df.columns

Index(['Date', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume', 'title',
       'content', 'sentiment_score', 'cleaned_content'],
      dtype='object')

In [39]:
merged_df.to_csv('multimodal_XOM_all.csv')